# SI-GCN input preparation (Pisa emissions)

This notebook prepares the Pisa emission triples and train/validation/test
splits used with the SI-GCN baseline reported in the INTSYS 2024 paper. It
does **not** train the SI-GCN model.

The model implementation is kept in a separate fork:
[github.com/baygaliyev/flow-imputation](https://github.com/baygaliyev/flow-imputation),
itself a fork of [susurrant/flow-imputation](https://github.com/susurrant/flow-imputation).

Input data are restricted and are not included in this repository. Input
paths below follow the restricted Google Drive layout.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass  # running locally: set DATA_DIR explicitly below

In [ ]:
CITY = "Pisa"
DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/phd_data")
OUTPUT_DIR = DATA_DIR / "si_gcn"

MAX_CO2_PER_M = 100.0  # Outlier cutoff.
TRAIN_RATIO = 0.8
VAL_RATIO = 0.10
SCALE_FACTOR = 1000.0  # Scaling applied before the split.
RANDOM_STATE = None    # set an integer for reproducible reruns
SAVE = True            # set False to inspect without writing files

if SAVE:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Emission triples and entities

In [ ]:
emissions = pd.read_csv(DATA_DIR / f"{CITY.lower()}_embs_n_co2m.csv")
emissions = emissions.loc[emissions["CO_2/m"] < MAX_CO2_PER_M].copy()
emissions["relation"] = 0

roads = emissions[["start_point", "relation", "end_point", "CO_2/m"]].copy()
entities = pd.DataFrame(
    {
        "node": pd.concat(
            [roads["start_point"], roads["end_point"]], ignore_index=True
        ).drop_duplicates()
    }
)

print(f"{len(roads)} emission triples; {len(entities)} unique nodes")
roads.head()

In [ ]:
if SAVE:
    entities.to_csv(DATA_DIR / f"{CITY.lower()}_entities_full.csv", index=False)
    roads.to_csv(OUTPUT_DIR / f"si_gcn_emissions_{CITY.lower()}.csv", index=False)

## Train / validation / test split

The split is 80% / 10% / 10%; `CO_2/m` is scaled by 1000 before splitting.

In [ ]:
split_data = roads.copy()
split_data["CO_2/m"] = split_data["CO_2/m"] * SCALE_FACTOR

shuffled = split_data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
n_train = int(len(shuffled) * TRAIN_RATIO)
n_val = int(len(shuffled) * VAL_RATIO)

train = shuffled.iloc[:n_train]
val = shuffled.iloc[n_train : n_train + n_val]
test = shuffled.iloc[n_train + n_val :]

print(f"train={len(train)}, val={len(val)}, test={len(test)}")

In [ ]:
if SAVE:
    stem = f"si_gcn_emissions_{CITY.lower()}"
    train.to_csv(OUTPUT_DIR / f"{stem}_train.csv", index=False)
    val.to_csv(OUTPUT_DIR / f"{stem}_val.csv", index=False)
    test.to_csv(OUTPUT_DIR / f"{stem}_test.csv", index=False)

## Notes

- The output triples use `start_point`, `relation`, `end_point`, `CO_2/m`,
  matching the SI-GCN fork's triple-file convention.
- Only the inputs needed for SI-GCN are included here.
- Model training and evaluation are performed in the separate fork, not here.